# Lab 08: What Does Boosting Learn About Pokémon?

Begin with the supplied data preparation and decision-tree baselines.
Fit **XGBoost** and **CatBoost** for the first legendary-status comparison,
then choose one library for the rest of the lab. Study class weighting and
predictions across boosting rounds, apply the chosen library to primary-type
classification, and finish with held-out evaluation.

**Total:** 45 points. Replace the TODOs with executable code and complete the
nine written responses. Keep your fitted models for later parts.

Read the background and variable guide below. `README.md` covers setup and submission.


## Pokémon background

Pokémon are fictional creatures with battle statistics and one or two elemental
**types**. Types affect resistance and vulnerability to attacks. Each row here
describes a species or form, rather than an individual trained Pokémon.
**Base statistics** describe underlying battle attributes; an individual's
level and training also affect its actual statistics. Use the supplied
`is_legendary` label as the definition of legendary status for this lab.

### Essential variable guide

| Group | Essential meaning and columns |
| --- | --- |
| Battle profile | `hp`, `attack`, `defense`, `sp_attack`, `sp_defense`, and `speed` describe endurance, physical and special offense and defense, and move order. `base_total` is their sum. |
| Other numeric attributes | `height_m` and `weight_kg` measure size. `percentage_male`, `base_egg_steps`, `base_happiness`, `capture_rate`, `experience_growth`, and `generation` describe species characteristics and game mechanics. `capture_rate` is a catchability parameter, not a probability. |
| Types and matchups | `type1` is the primary type; `type2` is a secondary type when recorded. Each `against_*` column is an incoming damage multiplier: 1 is neutral, values above 1 indicate vulnerability, values between 0 and 1 indicate resistance, and 0 indicates immunity. |
| Labels and descriptive fields | `is_legendary` is the binary label. `name` identifies entries during error analysis. Other descriptive fields include `japanese_name`, `pokedex_number`, `classfication`, and `abilities`. |

Physical and special moves use different offensive and defensive statistics.
The [official battle guide](https://diamondpearl.pokemon.com/en-gb/trainersguide/fundamentals/battling/),
[PokéAPI species documentation](https://pokeapi.co/docs/v2#pokemon-species), and
[type relationships](https://pokeapi.co/docs/v2#types) provide background.
The lab uses the supplied historical CSV.


## Data and evaluation protocol

The notebook loads the checked-in snapshot described in
[data/README.md](data/README.md) and applies the supplied row filters.
Data preparation, splits, decision-tree baselines, and reporting helpers are
provided. Your work is to fit the boosting models and interpret their results.
Keep the source CSV unchanged.

- **Training:** fit models and learned preprocessing; calculate class weights
  from these labels only.
- **Validation:** choose the library and model, apply early stopping, and
  investigate errors and feature importance.
- **Test:** make predictions and evaluate performance only in Part 7, after
  recording the model choices.

The shared preprocessing removes primary-flying entries and Minior.
Each task has its own fixed split, shared by every model for that task.
Within each class, about 60% of rows are used for training and 20% each for
validation and test. The primary-type task contains 17 classes.
These splits evaluate new entries under this protocol; they do not specifically
hold out evolutionary families or game generations.

The legendary task uses eight numeric predictors: the six battle statistics,
height, and weight. Binary classification uses a fixed probability cutoff of
0.5. The type task also includes categorical `generation`: XGBoost uses
`enable_categorical=True`, and CatBoost receives categorical column names.
The supplied baselines learn imputation and encoding from training rows.

After Part 2, keep `CHOSEN_LIBRARY` fixed for all later booster fits.


Run the supplied cells as written. Replace each commented TODO with working
code before submitting the completed notebook.


In [ ]:
import time
from importlib.metadata import version

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display
from catboost import CatBoostClassifier
from xgboost import XGBClassifier
from sklearn.metrics import (
    ConfusionMatrixDisplay, accuracy_score, classification_report,
    confusion_matrix, f1_score,
)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import LabelEncoder
from sklearn.tree import DecisionTreeClassifier

from lab_helpers import (
    RANDOM_STATE, TYPE_NAMES, BATTLE_STATS, LEGENDARY_FEATURES,
    load_data, make_features, categorical_columns, make_splits, split_counts,
    baseline_preprocessor, binary_metrics, case_table,
    stage_probabilities, loss_curve,
)

pd.set_option("display.max_columns", 16)
plt.rcParams.update({"figure.figsize": (8, 5), "axes.grid": True})
METRIC_TOLERANCE = 1e-12
EXCLUDED_PRIMARY_TYPE = "flying"
EXCLUDED_NAMES = {"Minior"}
raw = load_data()
# Apply the supplied row filters before both modeling tasks.
raw = raw.loc[
    (raw["type1"] != EXCLUDED_PRIMARY_TYPE) & ~raw["name"].isin(EXCLUDED_NAMES)
].copy()
assert EXCLUDED_PRIMARY_TYPE not in set(raw["type1"])
assert not raw["name"].isin(EXCLUDED_NAMES).any()
display(pd.Series({name: version(name) for name in ["xgboost", "catboost", "scikit-learn", "pandas"]}, name="installed version"))


## Part 1: Inspect the data and supplied baselines

### 1(a). Inspect the preprocessed data

The supplied cell reports dimensions, class counts and percentages,
majority-class accuracy, missing values, and primary-type counts.
Review the output; the Part 1 questions appear after the baseline.


In [ ]:
row_count, column_count = raw.shape
display(pd.Series({"rows": row_count, "columns": column_count}, name="dataset"))

legendary_counts = (
    raw["is_legendary"].astype(int).value_counts().reindex([0, 1], fill_value=0)
)
legendary_summary = pd.DataFrame({
    "count": legendary_counts,
    "percent": 100 * legendary_counts / row_count,
})
legendary_summary.index = legendary_summary.index.map(
    {0: "nonlegendary", 1: "legendary"}
)
display(legendary_summary)
display(pd.Series(
    {"majority_baseline_accuracy": legendary_counts.max() / row_count}
))

missing_counts = raw.isna().sum().rename("missing_count").to_frame()
display(missing_counts)

primary_type_counts = (
    raw["type1"].value_counts().rename_axis("primary_type").to_frame("count")
)
display(primary_type_counts)


### 1(b). Check the aggregate statistic

The supplied cell checks whether `base_total` equals the sum of `BATTLE_STATS`.
The legendary predictors retain the six components and exclude this aggregate.
A blank `type2` means no secondary type is recorded; a missing height or weight
means the measurement is unknown.


In [ ]:
component_total = raw[list(BATTLE_STATS)].sum(axis=1)
total_matches = raw["base_total"].eq(component_total)
display(pd.Series({
    "rows_checked": len(raw),
    "all_base_totals_match": bool(total_matches.all()),
    "mismatch_count": int((~total_matches).sum()),
}))
if not total_matches.all():
    display(pd.DataFrame({
        "base_total": raw.loc[~total_matches, "base_total"],
        "component_sum": component_total.loc[~total_matches],
    }))


### 1(c). Inspect the supplied legendary split

We use `make_features(raw, "legendary")` for eight predictors: `hp`, `attack`,
`defense`, `sp_attack`, `sp_defense`, `speed`, `height_m`, and `weight_kg`.
The question is whether battle profile and body size can identify legendary
status. Game mechanics, types, matchups, identifiers, descriptive text, and
`base_total` are excluded.

The supplied code displays split sizes, class counts, and checks for overlapping
rows. It also creates an always-nonlegendary reference whose constant probability
is the training proportion of legendary entries.


In [ ]:
y_l = raw["is_legendary"].astype(int)
X_l = make_features(raw, "legendary")
splits_l = make_splits(y_l)

X_train_l, y_train_l = X_l.loc[splits_l["train"]], y_l.loc[splits_l["train"]]
X_val_l, y_val_l = X_l.loc[splits_l["validation"]], y_l.loc[splits_l["validation"]]
X_test_l, y_test_l = X_l.loc[splits_l["test"]], y_l.loc[splits_l["test"]]
assert tuple(X_l.columns) == LEGENDARY_FEATURES

display(split_counts(y_l, splits_l))
display(pd.Series(
    {name: len(indices) for name, indices in splits_l.items()}, name="row_count",
).to_frame())
split_index_sets = {name: set(indices) for name, indices in splits_l.items()}
all_pairwise_disjoint = (
    split_index_sets["train"].isdisjoint(split_index_sets["validation"])
    and split_index_sets["train"].isdisjoint(split_index_sets["test"])
    and split_index_sets["validation"].isdisjoint(split_index_sets["test"])
)
all_rows_assigned = set().union(*split_index_sets.values()) == set(y_l.index)
display(pd.Series({
    "all_pairwise_disjoint": all_pairwise_disjoint,
    "all_rows_assigned": all_rows_assigned,
}))
majority_probability_l = np.full(len(y_val_l), y_train_l.mean())
majority_validation_metrics = binary_metrics(y_val_l, majority_probability_l)
display(majority_validation_metrics.rename("always-nonlegendary validation reference"))


### 1(d). Inspect the supplied single-tree baseline

The code fits `baseline_l`, a depth-three tree with training-only imputation.
Its depth matches each tree in the main boosting models. The `binary_metrics`
table reports accuracy, precision, recall, F1, average precision (AP), log loss,
legendary detections, misses, and false alarms.

AP summarizes the ranking of positive examples using probability scores;
here it means scikit-learn's `average_precision_score`. Higher AP is better,
and lower log loss is better. Classification scores and counts use the fixed
0.5 cutoff. Validation and test metrics remain unweighted when fitting uses weights.


In [ ]:
baseline_l = Pipeline([
    ("preprocess", baseline_preprocessor(X_train_l)),
    ("tree", DecisionTreeClassifier(max_depth=3, random_state=RANDOM_STATE)),
])
baseline_l.fit(X_train_l, y_train_l)
probabilities_l = {
    "Always nonlegendary": majority_probability_l,
    "Baseline": baseline_l.predict_proba(X_val_l)[:, 1],
}
validation_scores = {
    name: binary_metrics(y_val_l, probability)
    for name, probability in probabilities_l.items()
}
display(pd.DataFrame(validation_scores).T.rename_axis("main legendary validation"))


**Part 1 — data choices and baseline interpretation**

Use the baseline detections and false alarms to explain why accuracy alone
is insufficient. In the same short paragraph, explain the purpose of excluding
`base_total` and game mechanics and of estimating preprocessing and class
weights from training rows only.

Write your response here.


## Part 2: Compare XGBoost and CatBoost

### 2(a). Fit the first two boosters

Using the settings supplied below, fit both models on `X_train_l`, `y_train_l`:

- `xgb_plain`: use `XGB_SETTINGS` with `objective="binary:logistic"`,
  `eval_metric="logloss"`, and `early_stopping_rounds=40`.
  Fit with `eval_set=[(X_val_l, y_val_l)]` and `verbose=False`.
- `cb_plain`: use `CB_SETTINGS` with `loss_function="Logloss"`,
  `eval_metric="Logloss"`, `early_stopping_rounds=40`, `use_best_model=True`,
  and `cat_features=categorical_columns(X_train_l)`.
  Fit with `eval_set=(X_val_l, y_val_l)`.

Each model can fit up to 400 rounds. Early stopping monitors validation log
loss and stops after 40 rounds without improvement. Keep the supplied settings
unless a later task changes them.

Time each fit with `time.perf_counter()`. Report its fit time and selected number
of rounds: `xgb_plain.best_iteration + 1` or `cb_plain.get_best_iteration() + 1`.
Both iteration indices start at zero.


In [ ]:
XGB_SETTINGS = dict(
    n_estimators=400, max_depth=3, learning_rate=0.05,
    subsample=0.9, colsample_bytree=0.9, reg_lambda=1.0,
    tree_method="hist", device="cpu", enable_categorical=True,
    importance_type="gain", n_jobs=2, random_state=RANDOM_STATE,
)
CB_SETTINGS = dict(
    iterations=400, depth=3, learning_rate=0.05, l2_leaf_reg=3.0,
    task_type="CPU", thread_count=2, random_seed=RANDOM_STATE,
    allow_writing_files=False, verbose=False,
)

# TODO 2(a): Instantiate, time, and fit xgb_plain and cb_plain as specified.
# XGBClassifier(**XGB_SETTINGS, objective=..., eval_metric=..., early_stopping_rounds=40)
# CatBoostClassifier(**CB_SETTINGS, loss_function=..., eval_metric=...,
#                    cat_features=..., early_stopping_rounds=40, use_best_model=True)


### 2(b). Compare the results and choose a library

Obtain legendary probabilities with `predict_proba(X_val_l)[:, 1]`.
Compare `binary_metrics` for the majority reference, `baseline_l`,
`xgb_plain`, and `cb_plain`. For each model, report legendary entries found
out of 14, missed legendary entries, and nonlegendary false alarms.

Set `CHOSEN_LIBRARY` to `"XGBoost"` or `"CatBoost"`. Keep the corresponding
existing fitted model as `chosen_plain`, and keep its validation probabilities.
Use that library for every later fit, including primary-type prediction.
Choose using validation results and your experience with the interfaces;
test-set performance must not enter this choice.


In [ ]:
# TODO 2(b): Compare validation metrics and detection counts for the first fits.
# CHOSEN_LIBRARY = "XGBoost"  # or "CatBoost"
# chosen_plain = {"XGBoost": xgb_plain, "CatBoost": cb_plain}[CHOSEN_LIBRARY]
# Retain chosen_plain and its validation probabilities.


**Part 2 — comparison and library choice**

Briefly explain what AP and detection counts show about the two boosters
relative to the single tree. State which library you will continue with and why.

Write your response here.


## Part 3: Weight the minority class

Calculate `positive_weight = n_nonlegendary_train / n_legendary_train` from
`y_train_l`. Keep `chosen_plain` and fit a new object, `chosen_weighted`,
using your library's original settings plus:

- XGBoost: `scale_pos_weight=positive_weight`.
- CatBoost: `class_weights=[1.0, positive_weight]` and
  `eval_metric="Logloss:use_weights=false"` to monitor unweighted validation loss.

Use the same rows and early-stopping setup as in Part 2. The unweighted
CatBoost fit keeps `eval_metric="Logloss"`.

Compare unweighted validation metrics for `chosen_plain` and `chosen_weighted`
at the fixed 0.5 cutoff. Class weights change the fitting loss; weighted
scores need not be calibrated probabilities under the original prevalence.


In [ ]:
# TODO 3: Calculate the weight from y_train_l and fit chosen_weighted.
# Report unweighted validation metrics and counts beside chosen_plain.


**Part 3 — effects of weighting**

Describe the effect of weighting on this validation split. Use detections,
misses, false alarms, and AP or F1 to explain the benefit and cost of the change.

Write your response here.


## Part 4: Follow boosting across rounds

### 4(a). Calculate one set of score corrections

For binary logistic loss, let $F(x)$ be the log-odds score and
$p=1/(1+e^{-F})$. The loss is

$$\ell(y,p)=-y\log p-(1-y)\log(1-p).$$

The negative gradient with respect to $F$ is $r=y-p$. Add loss, negative
gradient, and weighted negative gradient to the supplied three-row example.
Use `positive_weight` for rows with `y=1` and weight 1 for rows with `y=0`.


In [ ]:
gradient_examples = pd.DataFrame({
    "case": ["confident correct nonlegendary", "uncertain legendary", "confident missed legendary"],
    "y": [0, 1, 1],
    "p_legendary": [0.05, 0.45, 0.05],
})
# TODO 4(a): Add individual log loss, negative gradient, and weighted negative gradient.


**4(a) — prediction-dependent corrections**

Which row needs the largest correction, and should its score rise or fall?
Explain why the next tree fits real-valued corrections based on current
predictions. How does this differ from AdaBoost's explicit observation-weight updates?

Write your response here.


### 4(b). Compare two learning rates

For your chosen library, use `DIAGNOSTIC_XGB_SETTINGS` or
`DIAGNOSTIC_CB_SETTINGS` below to fit models at learning rates **0.03 and 0.30**.
Apply the training-derived class weight from Part 3 and fit all **300 rounds**.
Disable early stopping; CatBoost also needs `use_best_model=False`.
Store the models as `diagnostic_models[0.03]` and `diagnostic_models[0.30]`.

Use `loss_curve(model, X_train_l, y_train_l, X_val_l, y_val_l)` to obtain
unweighted training and validation log loss at recorded checkpoints.
Plot one panel per rate with a common vertical scale, and report the plotted
round with the smallest validation loss. Hold all other settings fixed between
the two fits.


In [ ]:
DIAGNOSTIC_XGB_SETTINGS = XGB_SETTINGS | dict(
    n_estimators=300, max_depth=4, subsample=1.0, colsample_bytree=1.0,
)
DIAGNOSTIC_CB_SETTINGS = CB_SETTINGS | dict(iterations=300, depth=4)

# TODO 4(b): Fit your chosen library at 0.03 and 0.30 with training class weights.
# Run all 300 rounds without early stopping; store diagnostic_models by rate.
# TODO: Compute loss_curve, plot both rates, and report minimum-loss checkpoints.


**4(b) — learning rate and overfitting**

Which rate improves faster initially? What evidence of overfitting, if any,
appears in the curves? Use the results to explain how early stopping would help.

Write your response here.


### 4(c). Follow individual Pokémon

Use `diagnostic_models[0.30]` to obtain validation legendary probabilities.
Call `case_table(raw, y_val_l, probability)` and display its five highest-loss
rows. From the full validation set, select three distinct rows: the lowest loss,
the highest loss, then the remaining row whose probability is closest to 0.5.
Keep their original row indices.

`stage_probabilities(model, X_val_l)` returns a table with observations in rows
and boosting-round counts in columns. Plot each selected row's probability of
its true class against rounds: use `p_legendary` for `y=1` and
`1 - p_legendary` for `y=0`. Label names and true labels, and keep the vertical
axis on [0, 1]. Individual loss is `-log(p_true_class)`.


In [ ]:
# TODO 4(c): Identify the three distinct validation cases and plot their
# true-class probability trajectories using the 0.30 diagnostic model.


**4(c) — uncertainty and individual trajectories**

Do the selected rows improve monotonically? Explain the difference between
an uncertain prediction and a confidently wrong one, and why falling average
training loss cannot guarantee improvement for each validation row.

Write your response here.


## Part 5: Predict primary type

### 5(a). Inspect the supplied preparation and baseline

We use `make_features(raw, "type")` with `type1` as target. This retains battle
statistics and other species attributes, and treats `generation` as categorical.
It excludes both type columns, `is_legendary`, every `against_*` matchup column,
and the common identifier/text fields.

The supplied code encodes the 17 types, displays class support by partition,
and fits `baseline_t`, another depth-three tree. The target codes identify
nominal classes; their numerical gaps do not measure differences between types.


In [ ]:
TYPE_NAMES_T = tuple(name for name in TYPE_NAMES if name in set(raw["type1"]))
assert len(TYPE_NAMES_T) == raw["type1"].nunique() == 17

type_encoder = LabelEncoder().fit(TYPE_NAMES_T)
y_t = pd.Series(
    type_encoder.transform(raw["type1"]),
    index=raw.index,
    name="primary_type_code",
)
X_t = make_features(raw, "type")
splits_t = make_splits(y_t)
X_train_t, y_train_t = X_t.loc[splits_t["train"]], y_t.loc[splits_t["train"]]
X_val_t, y_val_t = X_t.loc[splits_t["validation"]], y_t.loc[splits_t["validation"]]
X_test_t, y_test_t = X_t.loc[splits_t["test"]], y_t.loc[splits_t["test"]]

type_support = split_counts(y_t, splits_t)
type_support.index = type_encoder.inverse_transform(type_support.index.to_numpy())
display(type_support.rename_axis("primary_type"))
all_classes_present = bool(type_support.gt(0).all(axis=1).all())
display(pd.Series({"all_17_classes_in_each_partition": all_classes_present}))


baseline_t = Pipeline([
    ("preprocess", baseline_preprocessor(X_train_t)),
    ("tree", DecisionTreeClassifier(max_depth=3, random_state=RANDOM_STATE)),
])
baseline_t.fit(X_train_t, y_train_t)
baseline_type_prediction = baseline_t.predict(X_val_t)
type_baseline_metrics = pd.Series({
    "accuracy": accuracy_score(y_val_t, baseline_type_prediction),
    "macro_f1": f1_score(
        y_val_t, baseline_type_prediction, average="macro",
        labels=np.arange(len(TYPE_NAMES_T)), zero_division=0,
    ),
})
display(type_baseline_metrics.rename("type baseline validation"))


### 5(b). Fit a multiclass booster

Create a new `chosen_type` model using `CHOSEN_LIBRARY` and the corresponding
`XGB_SETTINGS` or `CB_SETTINGS`, with these task-specific parameters:

- XGBoost: `objective="multi:softprob"`, `num_class=len(TYPE_NAMES_T)`,
  `eval_metric="mlogloss"`, and `early_stopping_rounds=40`.
- CatBoost: `loss_function="MultiClass"`, `eval_metric="MultiClass"`,
  `cat_features=categorical_columns(X_train_t)`, `early_stopping_rounds=40`,
  and `use_best_model=True`.

Fit on `X_train_t`, `y_train_t` and use `X_val_t`, `y_val_t` for early stopping.
Use no binary class weight. Flatten predicted labels with
`np.asarray(...).ravel()` and compare validation accuracy and macro-F1 with
`baseline_t`. Macro-F1 is the unweighted mean of the 17 class-specific F1 scores.


In [ ]:
# TODO 5(b): Fit chosen_type using CHOSEN_LIBRARY and the type partitions.
# Compare accuracy and macro-F1 with the supplied type_baseline_metrics.


### 5(c). Inspect the errors

For `chosen_type`, show a `classification_report` with type names and class
support, plus a row-normalized validation confusion matrix. Use
`type_encoder.classes_` for the class order. Each matrix row represents a
true type and sums to 1.

Identify one type with low F1 and one frequently confused pair. Use raw
confusion counts and class supports when describing them. Also count how many
primary-type errors predict the recorded `type2`. Decode predicted codes to
type names and align them with the validation row indices before comparing.
`type2` remains excluded from the predictors.


In [ ]:
# TODO 5(c): Report class-specific scores and the named confusion matrix.
# Identify a low-F1 type and a common confusion pair using raw counts.
# Decode predictions with type_encoder.inverse_transform(...), then compare
# with raw.loc[y_val_t.index, "type2"] for the post-prediction analysis.


**Part 5 — multiclass comparison and errors**

Explain what accuracy and macro-F1 show about the booster and baseline.
Use one weak type and one confusion pair to interpret the errors. Why does a
prediction matching a recorded secondary type still miss the primary-type target?

Write your response here.


## Part 6: Interpret feature importance

Pair `feature_importances_` with the corresponding training-frame columns:
`X_train_l.columns` for `chosen_weighted` and `X_train_t.columns` for
`chosen_type`. Draw two labeled horizontal-bar panels: all eight legendary
predictors and the top 10 type predictors. List the top three features for each task.

XGBoost reports normalized **gain**; CatBoost reports **PredictionValuesChange**.
XGBoost's scores summarize all stored trees, which can extend beyond its
best-iteration prediction prefix. Correlated inputs can share or substitute
for one another's importance.


In [ ]:
# TODO 6: Plot the two importance panels and list each task's top three features.


**Part 6 — feature interpretation**

What do the leading features suggest about the two tasks? Explain one
limitation of using these global rankings to explain predictions.

Write your response here.


## Part 7: Record choices and evaluate the test sets

### 7(a). Record the final models

Within your chosen library, compare `chosen_plain` and `chosen_weighted` by
validation AP. Treat values within `METRIC_TOLERANCE` of the maximum as tied
and prefer the unweighted fit when tied. Keep the winner as
`selected_legendary_model`. The other initial library and the diagnostic fits
are excluded.

Keep `chosen_type` as the type model. Before test predictions, record each
model's name, validation score (AP or macro-F1), and selected number of rounds
in a table. Keep both fits without refitting, and use the fixed 0.5 cutoff
for legendary classification. The table is the response for this step.


In [ ]:
# TODO 7(a): Choose selected_legendary_model from chosen_plain and
# chosen_weighted using validation AP. Record it and chosen_type before testing.


### 7(b). Evaluate once

Evaluate `selected_legendary_model` and `baseline_l` on `X_test_l`, `y_test_l`.
Report `binary_metrics` at 0.5, including detections, misses, and false alarms,
and draw the booster's labeled confusion matrix. Evaluate `chosen_type` and
`baseline_t` on `X_test_t`, `y_test_t` using accuracy and macro-F1.

Report the number of legendary test positives and the smallest type-test
class support, naming every type tied at that support. Keep the recorded
choices after seeing these results.


In [ ]:
# TODO 7(b): Evaluate the frozen legendary model and chosen_type beside the
# supplied baselines. Report the metrics, confusion matrix, and test supports.


**Part 7 — final reflection**

Give three short evidence-based bullets:

1. Compare each booster with its baseline and identify the harder task.
2. Explain one result from weighting or boosting trajectories.
3. State a limitation using the test class supports and the recorded-choice protocol.

Write your response here.


## Submission check

- All required TODOs have been replaced with executable code, and the nine required
  written responses are complete.
- Tables identify validation versus test results; figures have readable titles,
  axes, legends, type names, and class labels as appropriate.
- Class weights, early stopping, and model selection use
  the specified partitions. Test results do not determine your choices.
- The supplied row filters are applied, and the source CSV, helper, and
  environment files remain unchanged.
- Restart the kernel and run the completed notebook from top to bottom. Commit
  `lab-08.ipynb`; keep `uv.lock`, `.venv`, and model files out of the submission.

## References

- Rounak Banik, [The Complete Pokémon Dataset](https://www.kaggle.com/datasets/rounakbanik/pokemon).
- [XGBoost categorical inputs](https://xgboost.readthedocs.io/en/release_3.1.0/tutorials/categorical.html),
  [scikit-learn interface and early stopping](https://xgboost.readthedocs.io/en/release_3.1.0/python/sklearn_estimator.html),
  and [boosted-tree objective](https://xgboost.readthedocs.io/en/release_3.1.0/tutorials/model.html).
- CatBoost [categorical features](https://catboost.ai/docs/en/features/categorical-features),
  [class weights and parameters](https://catboost.ai/docs/en/references/training-parameters/common),
  [staged probabilities](https://catboost.ai/docs/en/concepts/python-reference_catboostclassifier_staged_predict_proba),
  and [built-in importance](https://catboost.ai/docs/en/concepts/fstr).
- Scikit-learn [average precision](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.average_precision_score.html).
